# EMD2 microbial metabolism of BBN

Run all cells with the default settings to reproduce Figures 3, 4 and S5. The notebook analyses published BCPN measurements and compares host and microbial sources in a compartmental model. Installation instructions are in the [README](README.md).

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
candidates = [HERE, *HERE.parents, HERE / "EMD2"]
ROOT = next((p for p in candidates
             if (p / "src" / "emd2_simulation" / "run.py").is_file()
             and (p / "data" / "emd2.sqlite").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from the EMD2 directory containing src/ and data/.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

## Run settings

Use `reference` for the manuscript figures, `quick` for a shorter model run, or `empirical` for the published-data analysis alone. Quick mode uses 40 parameter draws and omits the five synthetic profiles and the host-only sensitivity draws.

Use a new `RUN_NAME` to keep an exploratory run. Reusing a name replaces its generated files. `PARAMETER_OVERRIDES` changes assigned model parameters. `MAKE_FIGURE` and `EXPORT_TABLES` control which files are written.

In [ ]:
import hashlib
from IPython.display import Image, Markdown, display
from emd2_simulation.paths import DATABASE_PATH
from emd2_simulation.processing.store import table_catalog
from emd2_simulation.simulation.settings import SimulationSettings, prepare_parameters
from emd2_simulation.workflow import run_workflow
from emd2_simulation.reporting.tables import empirical_tables, simulation_tables, TABLE_DESCRIPTIONS

RUN_MODE = "reference"
RUN_NAME = "reproduced"
T_END_HOURS = 720.0
N_TIME_POINTS = 1441
N_ENSEMBLE = 400
N_NULL_AUDIT = 40
ENSEMBLE_SEED = 20260825
NULL_AUDIT_SEED = 20260826
RUN_IDENTIFIABILITY = True
RUN_ENSEMBLE = True
N_BOOTSTRAP = 20000
PARAMETER_OVERRIDES = {}

MAKE_FIGURE = True
EXPORT_TABLES = True
SHOW_FULL_LEGENDS = False
SELECTED_TABLES = ["empirical_consortium", "empirical_human_culture",
                   "model_endpoints", "model_checks", "model_sensitivity_endpoints"]

if RUN_MODE not in {"reference", "quick", "empirical"}:
    raise ValueError("Choose reference, quick or empirical mode.")
if not isinstance(RUN_NAME, str) or not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be one folder name.")
if RUN_MODE == "quick":
    N_ENSEMBLE, N_NULL_AUDIT, RUN_IDENTIFIABILITY = 40, 0, False
elif RUN_MODE == "empirical":
    RUN_ENSEMBLE, RUN_IDENTIFIABILITY, N_NULL_AUDIT = False, False, 0
if RUN_MODE != "reference" and RUN_NAME == "reproduced":
    RUN_NAME = RUN_MODE
parameters = prepare_parameters(PARAMETER_OVERRIDES)
settings = SimulationSettings(
    t_end_hours=T_END_HOURS, n_time_points=N_TIME_POINTS, n_ensemble=N_ENSEMBLE,
    n_null_audit=N_NULL_AUDIT, ensemble_seed=ENSEMBLE_SEED,
    null_audit_seed=NULL_AUDIT_SEED, run_identifiability=RUN_IDENTIFIABILITY,
    run_ensemble=RUN_ENSEMBLE, run_model=RUN_MODE != "empirical", n_bootstrap=N_BOOTSTRAP)
OUTDIR = ROOT / "outputs" / RUN_NAME
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Run mode:", RUN_MODE, "| output folder:", OUTDIR)

## Source data

The SQLite database contains 1,820 rows from the published workbook and 2,957 LC–MS acquisition records. Animals and donors are the biological units for the treatment comparisons. Acquisition records describe analytical coverage. The [data notes](data/THIRD_PARTY.md) give the units, exclusions and source citations.

In [ ]:
display(table_catalog())

## Run the analysis

The empirical analysis resamples animals or donors to estimate treatment contrasts. The model compares six arms under assigned kinetics. Its sensitivity ranges describe how the predictions vary across parameter draws.

In [ ]:
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter

log_path = OUTDIR / "emd2_run.log"
print("Progress log:", log_path)
started = perf_counter()
with log_path.open("w", encoding="utf-8") as log:
    with redirect_stdout(log), redirect_stderr(log):
        experiment = run_workflow(
            OUTDIR, settings, params=parameters,
            make_figure=MAKE_FIGURE, make_tables=EXPORT_TABLES)
empirical = experiment.empirical
summary = experiment.summary
simulated = experiment.simulation
tables = {**empirical_tables(empirical),
          **simulation_tables(summary, simulated.nominal if simulated else None,
                              simulated.time if simulated else None)}
print(f"Completed in {perf_counter() - started:.1f} seconds")
print("Published-data bootstrap draws:", empirical["n_bootstrap"])
if summary is not None:
    diagnostics = summary["diagnostics"]
    print(f"Internal checks: {diagnostics['checks_passed']}/{diagnostics['checks_evaluated']} evaluated; {diagnostics['checks_total']} registered")
    if diagnostics["skipped_checks"]:
        display(diagnostics["skipped_checks"])
    if simulated.ensemble is not None:
        print(f"Model ensemble: {simulated.ensemble['n_ok']}/{simulated.ensemble['n_requested']} accepted draws")
    if experiment.exit_code:
        print("See the model_checks table for failed checks.")

## Figures

Figures 3 and S5 report the published measurements. Figure 4 compares sources that can yield the same simulated urinary concentration. The first acute mouse samples were collected at one hour, so they cannot establish which compartment accumulated BCPN first.

In [ ]:
if MAKE_FIGURE:
    def show_emd2_figure(stem):
        caption = OUTDIR / f"{stem}_legend.md"
        title, _, body = caption.read_text(encoding="utf-8").partition("\n")
        display(Markdown("### " + title.lstrip("# ")))
        display(Image(filename=str(OUTDIR / f"{stem}.png"), width=1100))
        if SHOW_FULL_LEGENDS:
            display(Markdown(body.strip()))
        else:
            display(Markdown(f"[Figure caption]({caption.relative_to(ROOT).as_posix()})"))

    show_emd2_figure("emd2_empirical")
    if simulated is not None:
        show_emd2_figure("emd2_mechanistic")
    show_emd2_figure("emd2_empirical_controls")

## Numerical results

`SELECTED_TABLES` controls the previews. The `tables` dictionary contains the computed data frames, including when CSV export is disabled. The observable dictionary gives the model units. The model computes host and microbial contributions from assigned kinetics. The assay measures total BCPN.

In [ ]:
for name in SELECTED_TABLES:
    if name not in TABLE_DESCRIPTIONS:
        raise ValueError(f"Unknown table {name!r}; choose from {list(TABLE_DESCRIPTIONS)}")
    if name not in tables:
        continue
    display(Markdown(f"### {name.replace('_', ' ').capitalize()}"))
    frame = tables[name]
    if name == "model_endpoints":
        wanted = ["c_BCPN_cecum", "c_BCPN_plasma", "c_BCPN_urine", "AUC_uro", "micro_fraction", "ratio_cecum"]
        display(frame.loc[frame.observable.isin(wanted)].pivot(index="arm", columns="observable", values="value"))
    else:
        display(frame)

## Reproducibility checks

Verify the input checksum and bootstrap count. The reference run also checks all 13 model criteria, both sensitivity samples and the five synthetic profiles.

In [ ]:
assert experiment.metadata["input_database_sha256"] == hashlib.sha256(DATABASE_PATH.read_bytes()).hexdigest()
assert empirical["n_bootstrap"] == N_BOOTSTRAP
if summary is not None and experiment.metadata["reference_compatible"]:
    diagnostics = summary["diagnostics"]
    assert diagnostics["complete"]
    assert diagnostics["checks_passed"] == diagnostics["checks_total"] == 13
    assert simulated.ensemble["n_ok"] == N_ENSEMBLE
    assert simulated.ensemble["null_cost"]["n_ok"] == N_NULL_AUDIT
    assert len(summary["identifiability"]["sets"]) == 5
for path in [*experiment.paths, log_path]:
    assert path.is_file() and path.stat().st_size > 0, path
print(f"Checks passed. Results are in {OUTDIR.relative_to(ROOT)}/")